### Table of Contents

- [01_dataset_exploration.ipynb](#01_dataset_explorationipynb)
- [Initial Filters](#initial-filters)
- [Libraries](#libraries)
- [Definitions](#definitions)
- [Import Data Set](#import-data-set)
- [Filter Data Set](#filter-data-set)
- [Summary](#summary)
  - [Average ELO (Black/White)](#average-elo-blackwhite)
  - [average ELC (Combined)](#average-elc-combined)
  - [Results](#results)
  - [TimeControl Counts](#timecontrol-counts)
  - [Ply Count and NA Ply Count](#ply-count-and-na-ply-count)
  - [Total Observations](#total-observations)
  - [Average Observations Per Game](#average-observations-per-game)
- [Findings](#phase-2-findings)

### 01_dataset_exploration.ipynb

1. connect to Lichess
2. stream data
3. inspect schema
4. inspect raw movetext
5. filter Elo >= 1800
6. take 100 games
7. convert that sample to Pandas
8. inspect ratings/results/time controls
9. next: parse the 100 games and measure plies

### Initial Filters

Lichess/standard-chess-games

WhiteElo >= 1800
BlackElo >= 1800
Result in {"1-0", "0-1", "1/2-1/2"}

take 100 games

### Libraries

datasets - Hugging Face Datasets Python library - https://huggingface.co/docs/datasets/index

data-access and dataset-processing library designed for ML datasets

API:
* load_dataset(...)
* dataset.filter(...)
* dataset.map(...)
* dataset.take(...)
* dataset.shuffle(...)

Common storage formats:
* Parquet
* CSV
* JSON
* Arrow

### Definitions

| Term          | Definition                                                                                             | Example                                        | Notes                                                                                                                                                                       |
| ------------- | ------------------------------------------------------------------------------------------------------ | ---------------------------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `TimeControl` | Number of initial seconds each player has on their clock + number of seconds added after each move     | `60+1`, `600+5`                                | `60+1` is 1 minute + 1 second increment per move and is a very fast game. `600+5` is 10 minutes + 5 seconds increment per move and allows substantially more thinking time. |
| `MoveText`    | PGN move sequence containing the numbered moves played by White and Black, followed by the game result | `1. e4 e6 2. d4 b6 3. a3 Bb7 ... 13. Qe8# 1-0` | Each move number normally contains White's move followed by Black's move. The final move may contain only one ply if the game ends after White's move.                      |
| `Ply`         | A single move made by one player                                                                       | `e4`, `e6`, `d4`, `b6`                         | One White move or one Black move = 1 ply. A normal numbered pair such as `1. e4 e6` contains 2 plies.                                                                       |


```
1. e4 e6
   │  │
   │  └─ ply 2 — Black
   └──── ply 1 — White

"1." = move number
"e4" = one ply
"e6" = one ply

1. e4 e6 = 2 plies
```

* each ply gives us one potential supervised training observation




### Import Data Set

In [ ]:
from datasets import load_dataset
import pandas as pd

DATASET_NAME = "Lichess/standard-chess-games"
MIN_ELO = 1800
SAMPLE_SIZE = 100

/mnt/c/working/public/chess-multiplayer/ml/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# This gives you a Hugging Face IterableDataset. Because streaming=True, you're not downloading the entire Lichess dataset into memory or disk first.

# Conceptually:

    # Lichess dataset
    #     ↓
    # IterableDataset
    #     ↓
    # rows retrieved as we iterate

games = load_dataset(
    DATASET_NAME,
    split="train",
    streaming=True,
)

games

IterableDataset({
    features: ['Event', 'Site', 'White', 'Black', 'Result', 'WhiteTitle', 'BlackTitle', 'WhiteElo', 'BlackElo', 'WhiteRatingDiff', 'BlackRatingDiff', 'UTCDate', 'UTCTime', 'ECO', 'Opening', 'Termination', 'TimeControl', 'movetext'],
    num_shards: 26138
})

In [6]:
games.features
# list(games.features)

{'Event': Value('string'),
 'Site': Value('string'),
 'White': Value('string'),
 'Black': Value('string'),
 'Result': Value('string'),
 'WhiteTitle': Value('string'),
 'BlackTitle': Value('string'),
 'WhiteElo': Value('int16'),
 'BlackElo': Value('int16'),
 'WhiteRatingDiff': Value('int16'),
 'BlackRatingDiff': Value('int16'),
 'UTCDate': Value('date32'),
 'UTCTime': Value('time32[ms]'),
 'ECO': Value('string'),
 'Opening': Value('string'),
 'Termination': Value('string'),
 'TimeControl': Value('string'),
 'movetext': Value('string')}

In [ ]:
# Returns one game from the IterableDataset

game = next(iter(games))

game

{'Event': 'Rated Classical game',
 'Site': 'https://lichess.org/j1dkb5dw',
 'White': 'BFG9k',
 'Black': 'mamalak',
 'Result': '1-0',
 'WhiteTitle': None,
 'BlackTitle': None,
 'WhiteElo': 1639,
 'BlackElo': 1403,
 'WhiteRatingDiff': 5,
 'BlackRatingDiff': -8,
 'UTCDate': datetime.date(2012, 12, 31),
 'UTCTime': datetime.time(23, 1, 3),
 'ECO': 'C00',
 'Opening': 'French Defense: Normal Variation',
 'Termination': 'Normal',
 'TimeControl': '600+8',
 'movetext': '1. e4 e6 2. d4 b6 3. a3 Bb7 4. Nc3 Nh6 5. Bxh6 gxh6 6. Be2 Qg5 7. Bg4 h5 8. Nf3 Qg6 9. Nh4 Qg5 10. Bxh5 Qxh4 11. Qf3 Kd8 12. Qxf7 Nc6 13. Qe8# 1-0'}

In [8]:
game["movetext"]

'1. e4 e6 2. d4 b6 3. a3 Bb7 4. Nc3 Nh6 5. Bxh6 gxh6 6. Be2 Qg5 7. Bg4 h5 8. Nf3 Qg6 9. Nh4 Qg5 10. Bxh5 Qxh4 11. Qf3 Kd8 12. Qxf7 Nc6 13. Qe8# 1-0'

### Filter Data Set

In [29]:
# Filters games based on minimum Elo and valid results, then takes a sample.

# This is also lazy. It doesn't immediately scan the dataset and produce a new materialized dataset.

# You've effectively defined:

# For each game encountered:

    # WhiteElo exists?
    #     ↓ yes
    # BlackElo exists?
    #     ↓ yes
    # WhiteElo >= 1800?
    #     ↓ yes
    # BlackElo >= 1800?
    #     ↓ yes
    # valid completed result?
    #     ↓ yes
    # KEEP

filtered_games = games.filter(
    lambda game: (
        game["WhiteElo"] is not None
        and game["BlackElo"] is not None
        and game["WhiteElo"] >= MIN_ELO
        and game["BlackElo"] >= MIN_ELO
        and game["Result"] in {"1-0", "0-1", "1/2-1/2"}
    )
)

# Iterate through the streamed dataset, apply my filter, and stop once 100 matching games have been obtained.

sample = list(filtered_games.take(SAMPLE_SIZE))

print(f'sample type: {type(sample)}')
print(f'sample length: {len(sample)}')
print(sample[0])  # Print the first game in the sample to inspect its structure.


sample type: <class 'list'>
sample length: 100
{'Event': 'Rated Bullet game', 'Site': 'https://lichess.org/rklpc7mk', 'White': 'Naitero_Nagasaki', 'Black': '800', 'Result': '0-1', 'WhiteTitle': None, 'BlackTitle': None, 'WhiteElo': 1824, 'BlackElo': 1973, 'WhiteRatingDiff': -6, 'BlackRatingDiff': 8, 'UTCDate': datetime.date(2012, 12, 31), 'UTCTime': datetime.time(23, 4, 57), 'ECO': 'B12', 'Opening': 'Caro-Kann Defense: Goldman Variation', 'Termination': 'Normal', 'TimeControl': '60+1', 'movetext': '1. e4 c6 2. Nc3 d5 3. Qf3 dxe4 4. Nxe4 Nd7 5. Bc4 Ngf6 6. Nxf6+ Nxf6 7. Qg3 Bf5 8. d3 Bg6 9. Ne2 e6 10. Bf4 Nh5 11. Qf3 Nxf4 12. Nxf4 Be7 13. Bxe6 fxe6 14. Nxe6 Qa5+ 15. c3 Qe5+ 16. Qe3 Qxe3+ 17. fxe3 Kd7 18. Nf4 Bd6 19. Nxg6 hxg6 20. h3 Bg3+ 21. Kd2 Raf8 22. Rhf1 Ke7 23. d4 Rxf1 24. Rxf1 Rf8 25. Rxf8 Kxf8 26. e4 Ke7 27. Ke3 g5 28. Kf3 Be1 29. Kg4 Bd2 30. Kf5 Bc1 31. Kg6 Kf8 32. e5 Bxb2 33. Kxg5 Bxc3 34. h4 Bxd4 35. h5 Bxe5 36. g4 Bb2 37. Kf5 Kf7 38. g5 Bc1 39. g6+ Ke7 40. Ke5 b5 41. Kd4 Kd6

In [ ]:
# load_dataset(..., streaming=True)
#         ↓
# games
# IterableDataset
# lazy — nothing substantial materialized

#         ↓

# games.filter(...)
#         ↓
# filtered_games
# another IterableDataset
# lazy — defines which rows should pass

#         ↓

# filtered_games.take(100)
#         ↓
# IterableDataset containing the first
# 100 qualifying records
# still iterable/lazy

#         ↓

# list(...)
#         ↓
# Python list
# MATERIALIZED IN MEMORY
# [
#     {game 1},
#     {game 2},
#     ...
#     {game 100}
# ]

#         ↓

# pd.DataFrame(sample)
#         ↓
# Pandas DataFrame
# 100 rows × dataset columns

df = pd.DataFrame(sample)

df.head()

,Event,Site,White,Black,Result,WhiteTitle,BlackTitle,WhiteElo,BlackElo,WhiteRatingDiff,BlackRatingDiff,UTCDate,UTCTime,ECO,Opening,Termination,TimeControl,movetext
0,Rated Bullet game,https://lichess.org/rklpc7mk,Naitero_Nagasaki,800,0-1,None,None,1824,1973,-6,8,2012-12-31,23:04:57,B12,Caro-Kann Defense: Goldman Variation,Normal,60+1,1. e4 c6 2. Nc3 d5 3. Qf3 dxe4 4. Nxe4 Nd7 5. ...
1,Rated Blitz game,https://lichess.org/vb3w3rmn,nichiren1967,chinokoli,1/2-1/2,None,None,1878,1940,2,-2,2012-12-31,23:04:28,B21,Sicilian Defense: McDonnell Attack,Normal,300+0,1. e4 c5 2. f4 d5 3. exd5 Qxd5 4. Nc3 Qd8 5. B...
2,Rated Classical game,https://lichess.org/iclkx584,Voltvolf,Marzinkus,1-0,None,None,1824,1811,11,-11,2012-12-31,23:10:00,C02,French Defense: Advance Variation #2,Normal,360+6,1. e4 e6 2. d4 d5 3. e5 c5 4. c3 Ne7 5. f4 cxd...
3,Rated Blitz game,https://lichess.org/v778e8mr,chinokoli,nichiren1967,1-0,None,None,1938,1880,9,-10,2012-12-31,23:11:15,D20,Queen's Gambit Accepted: Saduleto Variation,Normal,300+0,1. d4 d5 2. c4 dxc4 3. e4 g6 4. Bxc4 Bg7 5. Ne...
4,Rated Bullet game,https://lichess.org/0wn9o371,Ben_Dover,schutzstaffel,1-0,None,None,1876,1877,10,-14,2012-12-31,23:40:47,C00,French Defense: Normal Variation,Normal,60+0,1. e4 e6 2. d4 Ne7 3. Nf3 Ng6 4. Bg5 Be7 5. Bx...


In [11]:
columns = [
    "WhiteElo",
    "BlackElo",
    "Result",
    "UTCDate",
    "ECO",
    "Opening",
    "Termination",
    "TimeControl",
    "movetext",
]

df[columns].head()

,WhiteElo,BlackElo,Result,UTCDate,ECO,Opening,Termination,TimeControl,movetext
0,1824,1973,0-1,2012-12-31,B12,Caro-Kann Defense: Goldman Variation,Normal,60+1,1. e4 c6 2. Nc3 d5 3. Qf3 dxe4 4. Nxe4 Nd7 5. ...
1,1878,1940,1/2-1/2,2012-12-31,B21,Sicilian Defense: McDonnell Attack,Normal,300+0,1. e4 c5 2. f4 d5 3. exd5 Qxd5 4. Nc3 Qd8 5. B...
2,1824,1811,1-0,2012-12-31,C02,French Defense: Advance Variation #2,Normal,360+6,1. e4 e6 2. d4 d5 3. e5 c5 4. c3 Ne7 5. f4 cxd...
3,1938,1880,1-0,2012-12-31,D20,Queen's Gambit Accepted: Saduleto Variation,Normal,300+0,1. d4 d5 2. c4 dxc4 3. e4 g6 4. Bxc4 Bg7 5. Ne...
4,1876,1877,1-0,2012-12-31,C00,French Defense: Normal Variation,Normal,60+0,1. e4 e6 2. d4 Ne7 3. Nf3 Ng6 4. Bg5 Be7 5. Bx...


### Explore Data Set

In [12]:
df[["WhiteElo", "BlackElo"]].describe()

,WhiteElo,BlackElo
count,100.000000,100.000000
mean,1908.490000,1929.720000
std,91.798692,93.212268
min,1801.000000,1800.000000
25%,1845.250000,1856.000000
50%,1879.000000,1905.500000
75%,1955.500000,1973.250000
max,2180.000000,2185.000000


In [13]:
df["AverageElo"] = (
    df["WhiteElo"] + df["BlackElo"]
) / 2

df["AverageElo"].describe()

count     100.000000
mean     1919.105000
std        69.366116
min      1807.000000
25%      1872.375000
50%      1903.750000
75%      1954.375000
max      2070.000000
Name: AverageElo, dtype: float64

In [ ]:
df["Result"].value_counts()

Result
0-1        54
1-0        44
1/2-1/2     2
Name: count, dtype: int64

In [30]:
df["TimeControl"].head()

0     60+1
1    300+0
2    360+6
3    300+0
4     60+0
Name: TimeControl, dtype: str

In [32]:
df["TimeControl"].value_counts().head(15)

TimeControl
60+0     37
180+0    21
120+0    12
240+0     5
180+3     5
300+0     4
480+2     3
480+8     2
360+8     2
240+1     2
60+1      1
360+6     1
600+5     1
180+2     1
480+5     1
Name: count, dtype: int64

In Lichess PGN data, `TimeControl` describes the **chess clock**, usually as:

```text
initial_seconds + increment_seconds
```

For example:

```text
TimeControl = "600+5"
```

means:

```text
600 seconds initial time = 10 minutes
5 seconds added after each move
```

So that's **10+5 chess**.

Other examples:

| `TimeControl` | Meaning                  |
| ------------- | ------------------------ |
| `60+0`        | 1 minute, no increment   |
| `180+0`       | 3 minutes, no increment  |
| `300+3`       | 5 minutes + 3 sec/move   |
| `600+0`       | 10 minutes, no increment |
| `900+10`      | 15 minutes + 10 sec/move |
| `1800+0`      | 30 minutes, no increment |

The second number is an increment **per move by that player**, not a number of moves or turns.

For example, with:

```text
300+5
```

White starts with 300 seconds. After White makes `e4`, 5 seconds are added to White's remaining clock. After Black makes `e5`, 5 seconds are added to Black's remaining clock.

This could actually become relevant to our training-data selection. When we run:

```python
df["TimeControl"].value_counts().head(20)
```

we may find that our `>= 1800 Elo` sample contains a mixture of bullet, blitz, rapid, etc. We can then decide whether we want the CNN learning equally from **1-minute bullet moves and 15-minute rapid moves**, or whether V1 should use a narrower set of time controls.


In [14]:
len(game["movetext"].split())

39

In [15]:
import io
import chess.pgn

In [ ]:
def count_plies(movetext: str) -> int | None:
    pgn_text = f"""
[Event "?"]
[Site "?"]
[Date "????.??.??"]
[Round "?"]
[White "?"]
[Black "?"]
[Result "*"]

{movetext}
"""

    game = chess.pgn.read_game(io.StringIO(pgn_text))

    if game is None:
        return None

    return sum(1 for _ in game.mainline_moves())

In [ ]:
df["ply_count"] = df["movetext"].apply(count_plies)

df["ply_count"].isna().value_counts()

count    100.000000
mean      72.340000
std       26.139185
min       25.000000
25%       53.750000
50%       68.000000
75%       87.250000
max      148.000000
Name: ply_count, dtype: float64

In [20]:
df["ply_count"].describe()

count    100.000000
mean      72.340000
std       26.139185
min       25.000000
25%       53.750000
50%       68.000000
75%       87.250000
max      148.000000
Name: ply_count, dtype: float64

In [33]:
total_observations = int(df["ply_count"].sum())

total_observations

7234

In [34]:
average_observations_per_game = df["ply_count"].mean()

average_observations_per_game

np.float64(72.34)

In [23]:
for game_count in [100, 1_000, 10_000, 100_000]:
    estimated = game_count * average_observations_per_game
    print(
        f"{game_count:>7,} games → "
        f"~{estimated:,.0f} observations"
    )

    100 games → ~7,234 observations
  1,000 games → ~72,340 observations
 10,000 games → ~723,400 observations
100,000 games → ~7,234,000 observations


In [24]:
df[
    [
        "WhiteElo",
        "BlackElo",
        "Result",
        "TimeControl",
        "ply_count",
    ]
].sort_values("ply_count").head(10)

,WhiteElo,BlackElo,Result,TimeControl,ply_count
20,1803,1820,1-0,480+8,25
51,2061,1837,1-0,60+0,27
78,1915,1800,0-1,240+1,30
73,1992,2045,0-1,120+0,32
35,1891,1865,1-0,60+0,37
80,1814,1962,0-1,180+0,40
68,1908,2110,1-0,60+0,41
42,1823,1961,1-0,360+8,43
59,1813,2078,1-0,60+0,45
61,2125,1974,1-0,180+0,45


In [25]:
df[
    [
        "WhiteElo",
        "BlackElo",
        "Result",
        "TimeControl",
        "ply_count",
    ]
].sort_values("ply_count", ascending=False).head(10)

,WhiteElo,BlackElo,Result,TimeControl,ply_count
10,1889,1932,0-1,180+0,148
74,2004,2040,0-1,120+0,144
30,1858,1885,1-0,60+0,131
18,1993,1976,0-1,120+0,126
98,2113,1851,0-1,120+0,126
15,1851,1940,1/2-1/2,240+0,124
90,1921,1815,0-1,600+0,118
81,1807,1856,0-1,180+2,118
23,1854,1855,1-0,60+0,117
1,1878,1940,1/2-1/2,300+0,117


#### Summary

```text
DATASET_NAME = "Lichess/standard-chess-games"
MIN_ELO = 1800
SAMPLE_SIZE = 100
```

##### Average ELO (Black/White)

```python
df[["WhiteElo", "BlackElo"]].describe()
```

```
	WhiteElo	BlackElo
count	100.000000	100.000000
mean	1908.490000	1929.720000
std	91.798692	93.212268
min	1801.000000	1800.000000
25%	1845.250000	1856.000000
50%	1879.000000	1905.500000
75%	1955.500000	1973.250000
max	2180.000000	2185.000000
```

##### average ELC (Combined)

```python
df["AverageElo"] = (
    df["WhiteElo"] + df["BlackElo"]
) / 2
```

```
count     100.000000
mean     1919.105000
std        69.366116
min      1807.000000
25%      1872.375000
50%      1903.750000
75%      1954.375000
max      2070.000000
Name: AverageElo, dtype: float64
```

##### Results

```python
df["Result"].value_counts()
```

```
Result
0-1        54
1-0        44
1/2-1/2     2
Name: count, dtype: int64
```

##### TimeControl Counts

```python
df["TimeControl"].value_counts().head(15)
```

```
TimeControl
60+0     37
180+0    21
120+0    12
240+0     5
180+3     5
300+0     4
480+2     3
480+8     2
360+8     2
240+1     2
60+1      1
360+6     1
600+5     1
180+2     1
480+5     1
Name: count, dtype: int64
```

##### Ply Count and NA Ply Count

```python
df["ply_count"].isna().value_counts()
```

```
count    100.000000
mean      72.340000
std       26.139185
min       25.000000
25%       53.750000
50%       68.000000
75%       87.250000
max      148.000000
Name: ply_count, dtype: float64
```

```python
df["ply_count"].describe()
```

```
count    100.000000
mean      72.340000
std       26.139185
min       25.000000
25%       53.750000
50%       68.000000
75%       87.250000
max      148.000000
Name: ply_count, dtype: float64
```

##### Total Observations

```python
total_observations = int(df["ply_count"].sum())
```
```
7234
```

##### Average Observations Per Game

```python
average_observations_per_game = df["ply_count"].mean()
```
```
72.34
```


### Phase 2 Findings

```text
Source
  Lichess standard rated games

Initial population
  both players >= 1800 Elo

Finding #1
  sample is heavily weighted toward fast time controls
  → revisit time-control filtering before real training

Finding #2
  each game produces many supervised observations
  → training scale should be thought about in positions,
    not merely number of games

The initial exploration successfully streamed and filtered 100 games from
`Lichess/standard-chess-games` where both players had an Elo rating of at
least 1800.

All sampled games were parsed with `python-chess`.

The 100 games contained 7,234 plies, or an average of 72.34 plies per game.
Since each ply can produce one supervised `(position, target move)`
observation, the sample represents approximately 7,234 potential training
observations.

The sample was heavily weighted toward fast time controls. In particular,
70% of games were 3+0 or faster, including 37% at 1+0. Time control should
therefore be considered when defining the larger training population.

The sample is intended for pipeline validation and exploration rather than
statistical analysis. It consists of the first 100 qualifying streamed games
rather than a representative random sample.
```